In [ ]:
from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document

load_dotenv()   # .env 파일에 저장되어있는 api 키 가져오기

In [ ]:
# 모델 설정
model = init_chat_model("openai:gpt-5.6-luna")

# 벡터 저장소 설정
# 임베딩 및 저장
DB_PATH = "../data/k_ladder_2026"

# 임베딩 모델 설정
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# 저장된 벡터 DB 가져와야
load_vs = Chroma(
    collection_name="k_ladder_2026",
    embedding_function=embeddings,
    persist_directory=DB_PATH
)

In [ ]:
# 검색기
retreiver_mmr = load_vs.as_retriever(search_type="mmr",
                                     search_kwargs={"k": 5, 
                                                    "fetch_k": 50,
                                                    "lambda_mult" : 0.25})

In [ ]:
# RAG 로 붙여보기
SYSTEM_PROMPT = """
너는 공공 정책 안내 도우미다.
아래 자료를 참고해서 답해라. 
참고 자료에 없으면 "자료에 없음" 이라고 말해라
정확한 자격, 금액, 기한은 공고 확인이 필요하다고 꼭 덧붙여라.
답 끝에 참고한 페이지 번호를 [p.60] 과 같은 형식으로 표시해라.
"""

# 검색 결과 문서를 받았을 때 메타데이터와 내용을 합쳐서 text 로 반환하는 함수 작성
def format_docs(docs):
    context = ""

    for doc in docs:
        context += f"[p.{doc.metadata['page']}] \n {doc.page_content} \n\n"

    return context

In [ ]:
chain = retreiver_mmr | format_docs
chain

In [ ]:
# 확인용
chain.invoke("청년 월세 지원 정책 찾아줘")

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

# rag_prompt 완성해보기
rag_prompt = ChatPromptTemplate.from_messages([
    ('system', SYSTEM_PROMPT),
    ('human', "참고자료\n{context} \n질문{question}")
])

                            
rag_chain = (
    {"context" : ( retreiver_mmr | format_docs ), "question" :  RunnablePassthrough()} # question 은 검색기를 거쳐서 문서찾아서 context 키값의 벨류 
    | rag_prompt  # question 은 그대로 prompt 에 들어가야
    | model
    | StrOutputParser()
    )

rag_chain.invoke("청년 월세 지원 정책 찾아줘")

NameError: name 'SYSTEM_PROMPT' is not defined

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from pydantic import BaseModel, Field

# 구조화된 출력 받기
from pydantic import BaseModel, Field

class AnswerStyle(BaseModel):
    answer : str = Field(description="최종 답변")
    source : str = Field(description="출처")

structured_model = model.with_structured_output(AnswerStyle, method="json_mode")
                            
rag_chain = (
    {"context" : ( retreiver_mmr | format_docs ), "question" :  RunnablePassthrough()} # question 은 검색기를 거쳐서 문서찾아서 context 키값의 벨류 
    | rag_prompt  # question 은 그대로 prompt 에 들어가야
    | structured_model
    )

result = rag_chain.invoke("청년 월세 지원 정책 찾아줘")

In [ ]:
result.model_dump()

In [ ]:
model.invoke("신대방 삼거리 맛집 알려줘")

In [ ]:
structured_model.invoke("신대방 삼거리 맛집 알려줘").model_dump()

In [ ]:
# 초등학생도 풀 수 있는 RAG 질문 연습
practice_questions = [
    '이 문서는 어떤 정책을 설명하고 있나요?',
    '청년 월세 지원은 누구를 위한 제도인가요?',
    '신청할 때 확인해야 할 것은 무엇인가요?',
    '지원 내용에서 꼭 기억할 숫자는 무엇인가요?',
    '더 자세히 물어보려면 어디에 문의하면 되나요?',
]

for number, question in enumerate(practice_questions, start=1):
    print(f'문제 {number}: {question}')
    answer = rag_chain.invoke(question)
    print(answer)
    print('-' * 60)

## 연습 문제 확인표

각 답변을 보고 다음 세 가지를 확인해 보세요.

1. 질문에 바로 답했나요?
2. 문서에 없는 내용을 상상해서 말하지 않았나요?
3. 답변에 참고 페이지가 표시되었나요?

답변이 이상하면 검색 결과 개수 `k`를 3 또는 5로 바꾸고 다시 실행해 보세요.


In [ ]:
# 내가 만든 질문 하나로 다시 테스트하기
my_question = '청년 월세 지원을 받으려면 무엇을 먼저 확인해야 하나요?'
my_answer = rag_chain.invoke(my_question)
print(my_answer)

## 가장 간단한 RAG 실습

질문을 하면 관련 문서를 찾고, 그 문서를 읽은 AI가 답변합니다.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

simple_prompt = ChatPromptTemplate.from_template("""
아래 문서만 참고해서 질문에 답해줘.
문서에 답이 없으면 '문서에서 찾지 못했어요'라고 말해줘.

문서:
{context}

질문:
{question}
""")

simple_chain = (
    {
        'context': retreiver_mmr | format_docs,
        'question': RunnablePassthrough(),
    }
    | simple_prompt
    | model
    | StrOutputParser()
)

simple_chain.invoke('청년 월세 지원은 누구를 위한 제도인가요?')